In [ ]:
library(tidyverse)
library(pheatmap)

# ============================================================
# 1. Paths and analysis settings
# ============================================================
BASE_DIR <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/"

H_ROOT <- file.path(
  BASE_DIR,
  "03_process_rega/matrices"
)

METADATA_ROOT <- file.path(
  BASE_DIR,
  "01_prepare/data/metadata_by_group"
)

SELECTED_FILE <- file.path(
  BASE_DIR,
  "02_run_rega/pathology_association",
  "selected_module.txt"
)

OUT_DIR <- file.path(dirname(SELECTED_FILE), "pheatmap")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)



In [ ]:
OUT_DIR 

In [ ]:
NON_PFC_REGIONS<- c("AG", "EC", "HC", "MTC", "TH")
REGIONS <- c("AG", "EC", "HC", "MTC","PFC", "TH")
ALL_CELLTYPES <- c("Exc", "Inh", "Ast", "Oli", "OPC", "Mic_Immune")
PATHOLOGY_LEVELS <- c("nonAD", "earlyAD", "lateAD")

In [ ]:
# ============================================================
# 2. Read and parse selected module names
# ============================================================
# Expected examples:
# Ast_AG_M6
# Mic_Immune_EC_M16
selected_names <- readLines(SELECTED_FILE) |>
  trimws()

selected_names <- selected_names[selected_names != ""]

if (length(selected_names) == 0) {
  stop("No modules were found in selected_modules_gsea.txt")
}

# Parse from the end so that Mic_Immune is handled correctly.
selected_info <- tibble(selected_module = selected_names) |>
  mutate(
    module = str_extract(selected_module, "M[0-9]+$"),
    celltype_region = str_remove(selected_module, "_M[0-9]+$"),
    region = str_extract(celltype_region, "(AG|EC|HC|MTC|PFC|TH)$"),
    celltype = str_remove(celltype_region, paste0("_", region, "$"))
  )

if (
  any(is.na(selected_info$module)) ||
  any(is.na(selected_info$region)) ||
  any(is.na(selected_info$celltype))
) {
  stop("Some names in /project/zduren/durenlab/naqing_palmetto/REGA_separate/02_run_rega/pathology_association/figures/selected_modules.txt could not be parsed.")
}


# ============================================================
# 3. Metadata helper
# ============================================================
read_group_metadata <- function(celltype, region, warn_if_missing = TRUE) {
  celltype_region <- paste(celltype, region, sep = "_")
  metadata_file <- file.path(
    METADATA_ROOT,
    paste0(celltype_region, "_metadata.tsv")
  )

  if (!file.exists(metadata_file)) {
    if (warn_if_missing) {
      warning("Missing metadata file: ", metadata_file)
    }
    return(NULL)
  }

  metadata <- read.delim(
    metadata_file,
    sep = "\t",
    header = TRUE,
    check.names = FALSE,
    stringsAsFactors = FALSE
  ) |>
    select(ROSMAP_IndividualID, Pathology) |>
    mutate(
      ROSMAP_IndividualID = as.character(ROSMAP_IndividualID),
      Pathology = trimws(as.character(Pathology))
    ) |>
    filter(
      !is.na(ROSMAP_IndividualID),
      ROSMAP_IndividualID != "",
      Pathology %in% PATHOLOGY_LEVELS
    ) |>
    distinct()

  conflicting_pathology <- metadata |>
    distinct(ROSMAP_IndividualID, Pathology) |>
    count(ROSMAP_IndividualID, name = "n_pathology_labels") |>
    filter(n_pathology_labels > 1)

  if (nrow(conflicting_pathology) > 0) {
    stop(
      "Conflicting pathology labels were found in ",
      metadata_file,
      "."
    )
  }

  metadata |>
    distinct(ROSMAP_IndividualID, .keep_all = TRUE) |>
    mutate(
      celltype = celltype,
      region = region,
      celltype_region = celltype_region
    )
}

In [ ]:
# ============================================================
# 4. Define the donor cohort from the union of all non-PFC groups
# ============================================================
# Every donor appearing in at least one cell-type/region metadata file outside
# PFC is retained. A donor absent from another group will later receive NA for
# that module instead of being removed from the heatmap.
non_pfc_group_grid <- tidyr::crossing(
  celltype = ALL_CELLTYPES,
  region = NON_PFC_REGIONS
)

non_pfc_metadata <- pmap_dfr(
  non_pfc_group_grid,
  read_group_metadata
)

if (nrow(non_pfc_metadata) == 0) {
  stop("No non-PFC metadata could be read; the union cohort cannot be defined.")
}

pathology_conflicts <- non_pfc_metadata |>
  distinct(ROSMAP_IndividualID, Pathology) |>
  count(ROSMAP_IndividualID, name = "n_pathology_labels") |>
  filter(n_pathology_labels > 1)

if (nrow(pathology_conflicts) > 0) {
  stop(
    "Some donors have conflicting pathology labels across non-PFC metadata files: ",
    paste(pathology_conflicts$ROSMAP_IndividualID, collapse = ", ")
  )
}
non_pfc_union <- non_pfc_metadata |>
  distinct(ROSMAP_IndividualID, Pathology) |>
  mutate(
    Pathology = factor(Pathology, levels = PATHOLOGY_LEVELS)
  ) |>
  arrange(Pathology, ROSMAP_IndividualID)

union_individuals <- non_pfc_union$ROSMAP_IndividualID

write.table(
  non_pfc_union,
  file.path(OUT_DIR, "non_PFC_union_individuals.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)


In [ ]:
# ============================================================
# 5. Read one selected module
# ============================================================
read_one_module <- function(selected_module, module,
                            celltype_region, region, celltype) {
  H_file <- file.path(
    H_ROOT,
    celltype_region,
    paste0(celltype_region, "_H_module_by_sample.tsv")
  )

  if (!file.exists(H_file)) {
    warning("Missing H file: ", H_file)
    return(NULL)
  }

  metadata <- read_group_metadata(celltype, region)

  if (is.null(metadata)) {
    return(NULL)
  }

  # PFC is restricted to the donor union defined from non-PFC regions.
  # The same filtering is harmless for non-PFC groups and guarantees that all
  # selected modules use exactly the same final donor universe.
  metadata <- metadata |>
    filter(ROSMAP_IndividualID %in% union_individuals)

  H <- read.delim(
    H_file,
    sep = "\t",
    header = TRUE,
    check.names = FALSE,
    stringsAsFactors = FALSE
  )

  colnames(H)[1] <- "Module"
  H$Module <- as.character(H$Module)

  if (!module %in% H$Module) {
    warning(module, " was not found in: ", H_file)
    return(NULL)
  }

  module_values <- H |>
    filter(Module == module) |>
    pivot_longer(
      cols = -Module,
      names_to = "ROSMAP_IndividualID",
      values_to = "activity"
    ) |>
    transmute(
      ROSMAP_IndividualID = as.character(ROSMAP_IndividualID),
      activity = suppressWarnings(as.numeric(activity))
    ) |>
    filter(ROSMAP_IndividualID %in% union_individuals)

  # Keep every donor present in this group's metadata. Do not remove rows with
  # missing activity: missing values must remain NA in the final heatmap.
  metadata |>
    select(ROSMAP_IndividualID, Pathology) |>
    left_join(module_values, by = "ROSMAP_IndividualID") |>
    mutate(
      selected_module = selected_module,
      module = module,
      celltype_region = celltype_region,
      celltype = celltype,
      region = region
    ) |>
    select(
      selected_module,
      module,
      celltype_region,
      celltype,
      region,
      ROSMAP_IndividualID,
      Pathology,
      activity
    )
}


# ============================================================
# 6. Read all selected modules
# ============================================================
all_activity_observed <- pmap_dfr(selected_info, read_one_module)

if (nrow(all_activity_observed) == 0) {
  stop("No selected module activity could be read and matched to metadata.")
}

available_modules <- selected_info |>
  filter(selected_module %in% unique(all_activity_observed$selected_module))

# Build the complete module-by-union-donor grid. This is the key step that
# converts any missing donor/cell-type/region combination to an explicit NA.
complete_grid <- tidyr::crossing(
  selected_module = available_modules$selected_module,
  ROSMAP_IndividualID = union_individuals
)

module_annotations <- available_modules |>
  select(selected_module, module, celltype_region, celltype, region)

activity_values <- all_activity_observed |>
  select(selected_module, ROSMAP_IndividualID, activity) |>
  group_by(selected_module, ROSMAP_IndividualID) |>
  summarise(
    activity = {
      values <- unique(activity[!is.na(activity)])
      if (length(values) > 1) {
        stop(
          "Multiple activity values were found for ",
          first(selected_module),
          " and donor ",
          first(ROSMAP_IndividualID),
          "."
        )
      }
      if (length(values) == 0) NA_real_ else values[[1]]
    },
    .groups = "drop"
  )

all_activity <- complete_grid |>
  left_join(module_annotations, by = "selected_module") |>
  left_join(activity_values, by = c("selected_module", "ROSMAP_IndividualID")) |>
  left_join(
    non_pfc_union |>
      mutate(Pathology = as.character(Pathology)),
    by = "ROSMAP_IndividualID"
  )

write.table(
  all_activity,
  file.path(OUT_DIR, "selected_module_activity_complete_long_table.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE,
  na = "NA"
)


In [ ]:
missing_combinations <- all_activity |>
  filter(is.na(activity)) |>
  arrange(region, celltype, selected_module, Pathology, ROSMAP_IndividualID)

write.table(
  missing_combinations,
  file.path(OUT_DIR, "missing_module_individual_combinations.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE,
  na = "NA"
)

In [ ]:
# ============================================================
# 7. Safe clustering helpers
# ============================================================
make_safe_euclidean_hclust <- function(x, axis_name) {
  if (nrow(x) < 2) {
    return(FALSE)
  }

  distance <- dist(
    x,
    method = "euclidean"
  )

  if (any(!is.finite(distance))) {
    warning(
      "Clustering was disabled for ", axis_name,
      " because at least one pair had no comparable ",
      "non-missing values."
    )
    return(FALSE)
  }

  hclust(distance, method = "complete")
}


In [ ]:

# ============================================================
# 8. Draw the combined heatmap
# ============================================================
plot_combined <- function(plot_name = "All_regions_union_cohort") {
  region_modules <- available_modules$selected_module

  activity_matrix <- all_activity |>
    select(selected_module, ROSMAP_IndividualID, activity) |>
    pivot_wider(
      names_from = ROSMAP_IndividualID,
      values_from = activity
    ) |>
    # Keep pathology-ordered union donors in a deterministic column order.
    select(selected_module, all_of(union_individuals)) |>
    column_to_rownames("selected_module") |>
    as.matrix()

  storage.mode(activity_matrix) <- "numeric"

  # A row needs at least two observed values and non-zero variance to be
  # Z-scored. NA entries themselves remain NA.
  row_n_observed <- rowSums(!is.na(activity_matrix))
  row_sd <- apply(activity_matrix, 1, sd, na.rm = TRUE)
  invalid_modules <- rownames(activity_matrix)[
    row_n_observed < 2 | is.na(row_sd) | row_sd == 0
  ]

  if (length(invalid_modules) > 0) {
    warning(
      plot_name,
      ": removed modules with fewer than two observations or zero variance: ",
      paste(invalid_modules, collapse = ", ")
    )
    activity_matrix <- activity_matrix[
      !rownames(activity_matrix) %in% invalid_modules,
      ,
      drop = FALSE
    ]
  }

  if (nrow(activity_matrix) == 0) {
    stop(plot_name, ": no variable modules remained.")
  }

  # Explicit row Z-score that ignores NA without imputing it.
  row_mean <- rowMeans(activity_matrix, na.rm = TRUE)
  row_sd <- apply(activity_matrix, 1, sd, na.rm = TRUE)
  activity_z <- sweep(activity_matrix, 1, row_mean, FUN = "-")
  activity_z <- sweep(activity_z, 1, row_sd, FUN = "/")

  # Cap only observed plotted values; NA remains NA.
  activity_z_plot <- pmax(pmin(activity_z, 2.5), -2.5)

  annotation_col <- non_pfc_union |>
    select(ROSMAP_IndividualID, Pathology) |>
    mutate(Pathology = factor(Pathology, levels = PATHOLOGY_LEVELS)) |>
    column_to_rownames("ROSMAP_IndividualID")

  annotation_col <- annotation_col[colnames(activity_z_plot), , drop = FALSE]

  annotation_row <- available_modules |>
    filter(selected_module %in% rownames(activity_z_plot)) |>
    select(selected_module, Region = region, Celltype = celltype) |>
    distinct() |>
    column_to_rownames("selected_module")

  annotation_row <- annotation_row[rownames(activity_z_plot), , drop = FALSE]

  annotation_colors <- list(
    Pathology = c(
      "nonAD" = "#66AABB",
      "earlyAD" = "#EEAA55",
      "lateAD" = "#CC6677"
    ),
    Region = c(
      "AG" = "#8DA0CB",
      "EC" = "#FC8D62",
      "HC" = "#66C2A5",
      "MTC" = "#E78AC3",
      "PFC" = "#B3B3B3",
      "TH" = "#A6D854"
    ),
    Celltype = c(
      "Exc" = "#0072B2",
      "Inh" = "#E69F00",
      "Ast" = "#009E73",
      "Oli" = "#D55E00",
      "OPC" = "#CC79A7",
      "Mic_Immune" = "#6B5B53",
      "Vasc" = "#56B4E9"
    )
  )

  heatmap_colors <- colorRampPalette(
    c("#3B6FB6", "#F7F7F7", "#C94C4C")
  )(101)

  heatmap_breaks <- seq(-2.5, 2.5, length.out = 102)
  # Cluster using the UNCAPPED Z-score matrix.
    #
    # Rows/modules:
    #   Spearman correlation distance groups modules according to
    #   similarity of their donor-level activity patterns.
    #
    # Columns/individuals:
    #   Euclidean distance preserves differences in standardized
    #   activity magnitude across disease-related modules.
    #
    # The capped matrix activity_z_plot is used only for visualization.

    row_clustering <- make_safe_euclidean_hclust(
      activity_z,
      "modules"
    )

    column_clustering <- make_safe_euclidean_hclust(
      t(activity_z),
      "individuals"
    )

    # ------------------------------------------------------------
    # Orient the donor dendrogram from lower overall selected-module
    # activity toward higher overall selected-module activity.
    #
    # IMPORTANT:
    # This does NOT change Euclidean distances, linkage heights, or
    # cluster membership. It only rotates branches of the existing
    # hierarchical tree for a more interpretable left-to-right display.
    # ------------------------------------------------------------
    if (!identical(column_clustering, FALSE)) {
      donor_mean_activity <- colMeans(
        activity_z,
        na.rm = TRUE
      )

      column_dend <- as.dendrogram(column_clustering)

      column_dend <- reorder(
        column_dend,
        wts = donor_mean_activity,
        agglo.FUN = mean
      )

      column_clustering <- as.hclust(column_dend)
    }

  figure_width <- max(9, ncol(activity_z_plot) * 0.18 + 3.5)
  figure_height <- max(4.5, nrow(activity_z_plot) * 0.32 + 2.5)

  pdf_file <- file.path(
    OUT_DIR,
    paste0(plot_name, "_selected_module_activity_pheatmap.pdf")
  )

  png_file <- file.path(
    OUT_DIR,
    paste0(plot_name, "_selected_module_activity_pheatmap.png")
  )

  heatmap_arguments <- list(
    mat = activity_z_plot,
    color = heatmap_colors,
    breaks = heatmap_breaks,
    na_col = "#D9D9D9",
    cluster_rows = row_clustering,
    cluster_cols = column_clustering,
    annotation_col = annotation_col,
    annotation_row = annotation_row,
    annotation_colors = annotation_colors,
    annotation_legend = TRUE,
    show_colnames = FALSE,
    show_rownames = TRUE,
    border_color = NA,
    fontsize = 17,
    fontsize_row = 16,
    main = "Selected module activity"
  )

  do.call(
    pheatmap,
    c(
      heatmap_arguments,
      list(
        filename = pdf_file,
        width = figure_width,
        height = figure_height
      )
    )
  )

  do.call(
    pheatmap,
    c(
      heatmap_arguments,
      list(
        filename = png_file,
        width = figure_width,
        height = figure_height
      )
    )
  )

  write.table(
    activity_matrix,
    file.path(OUT_DIR, paste0(plot_name, "_activity_matrix_raw.tsv")),
    sep = "\t",
    quote = FALSE,
    col.names = NA,
    na = "NA"
  )

  write.table(
    activity_z,
    file.path(OUT_DIR, paste0(plot_name, "_activity_matrix_row_Zscore.tsv")),
    sep = "\t",
    quote = FALSE,
    col.names = NA,
    na = "NA"
  )

  tibble(
    plot = plot_name,
    number_of_modules = nrow(activity_z_plot),
    number_of_union_individuals = ncol(activity_z_plot),
    number_of_missing_cells = sum(is.na(activity_z_plot)),
    pdf = pdf_file,
    png = png_file
  )
}


In [ ]:
# ============================================================
# 9. Run and save summary
# ============================================================
plot_summary <- plot_combined()

write.table(
  plot_summary,
  file.path(OUT_DIR, "pheatmap_summary.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = FALSE
)

print(plot_summary)
cat("\nDone. Results saved in:\n", OUT_DIR, "\n")
